# Line Radiative Transfer Pipeline

Interactive notebook using the high-level `LineRt` API.

In [ ]:
# Bootstrap the pipeline without installation.
# Set the path to your line_rt_pipeline checkout here:
import importlib.util, os;
_BOOT = os.path.expanduser(
    '~/Seafile/seafile_sync/code/line_rt_pipeline/line_rt_bootstrap.py' );
_spec = importlib.util.spec_from_file_location( 'line_rt_bootstrap', _BOOT );
lr = importlib.util.module_from_spec( _spec );
_spec.loader.exec_module( lr );

%matplotlib inline
LineRt          = lr.LineRt;
TransitionInfo  = lr.TransitionInfo;


## Quick start - Group 2 (explicit opacity)

No species data needed: provide `b_sca` and `mfp_i_sca_0` directly.

In [ ]:
AU = 1.49598e13;

rt = LineRt(
    n_cell     = ( 64, 2, 2 ),
    x_min      = ( -5, 0, 0 ), x_max = ( 5, 0.2, 0.2 ),
    unit_l0    = AU, unit_t0 = 1.0,

    b_sca       = 1e5,       # Doppler b [cm/s]
    mfp_i_sca_0 = 1e-13,     # inverse scattering MFP [cm^-1]
    mfp_i_abs_0 = 0.0,       # inverse absorption MFP [cm^-1]

    ph_mode    = 2,          # R_IIA, const-mem (production)
    n_step     = 10000, n_scat = 10000,
    n_cycles   = 1,
    visualize  = False,
);
rt.set_boundary( 'fre fre per per per per' );
rt.add_source( type = 'slab', x = -5.0, direction = '+x', \
               n_photon = 10000, luminosity = 1.0 );

results = rt.run( );


In [ ]:
# Plot spatial flux profile
from numpy import linspace, max as np_max;
import matplotlib.pyplot as plt;

flx = results.get( 'flx', [ ] );
nxc = int( results[ 'mesh' ][ 'n_cell' ][ 0 ] );
x_cell = linspace( -5, 5, nxc );

fig, ax = plt.subplots( figsize = ( 8, 4 ) );
if flx is not None and len( flx ) > 0:
    ax.plot( x_cell, flx[ 0, 0, : nxc ] );
    ax.set( xlabel = 'x [AU]', ylabel = 'flux', \
            title = 'Spatial flux (cycle 0)' );
else:
    ax.text( 0.5, 0.5, 'No flux data', transform = ax.transAxes, \
             ha = 'center' );
plt.show( );


## Group 1 - Species-based (CO J=1->0)

Uses LAMDA molecular data to compute cross-sections and populations.

In [ ]:
Lsun = 3.828e33;

rt2 = LineRt(
    n_cell     = ( 64, 2, 2 ),
    x_min      = ( -5, 0, 0 ), x_max = ( 5, 0.2, 0.2 ),
    unit_l0    = AU, unit_t0 = 1.0,

    transition_info = TransitionInfo( 'CO', 0 ),
    n_species       = 1e4, temperature = 100.0,

    ph_mode   = 2,
    n_step    = 20000, n_scat = 200000, n_cycles = 3,
    n_emission_max = 5,
    visualize = False,
);
rt2.set_boundary( 'fre fre per per per per' );
rt2.add_source( type = 'slab', x = -5.0, direction = '+x', \
                n_photon = 20000,
                flux = 0.8 * Lsun / ( 0.2 * 0.2 * AU * AU ), \
                wavelength = 2.6e-1 );

results2 = rt2.run( );


In [ ]:
# Summary
from numpy import max as np_max;

for k, res in enumerate( results2[ 'results' ] ):
    flx = res.get( 'flx' );
    n_esc = len( res.get( 'photons', { } ).get( 'vel', [ ] ) );
    fmax = '%.2e' % np_max( flx ) if flx is not None else 'N/A';
    print( '  Cycle %d: flx_max=%s, n_esc=%d' % ( k, fmax, n_esc ) );
